##  Test PF-controller neural network

**Note:** This notebook evaluates the trained particle-filter (PF) controller on the **test dataset** and visualizes its predictive behavior and uncertainty. No training or parameter updates are performed. Running this notebook is **optional** and not required to reproduce the main results.

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from src.models.particle_filter.core import ParticleFilter
from src.models.networks.pf_mlp import ParticleFilterMLP
import imageio
import matplotlib.pyplot as plt

from src.helpers.visualization import create_pf_prediciton_frame
from src.helpers.seed import set_global_seed

from experiment_config import (
    DegModel,SEED,LEAKY_SLOPE,HIDDEN_DIMS,PREDICTION_START_IDX,N_PARTICLES,PFNET_DIR,ESTIMATION_DIR,DEGR_MODEL_DIR,UNCERTAINTY_LEVEL
)


## Parameters

In [2]:
perform_name = "SmHPC"

performnet_dir = PFNET_DIR/perform_name
pred_dir = performnet_dir/ f'pred_ulevel{UNCERTAINTY_LEVEL}'
pred_dir.mkdir(parents=True, exist_ok=True)

In [3]:
set_global_seed(SEED)

## Load component (base) models

In [4]:
hi_df = pd.read_csv(ESTIMATION_DIR/'data_dev.csv')
dev_units = hi_df['unit'].astype(int).unique().tolist()
dev_units

[1, 2, 3, 4, 5, 6]

In [5]:
degmodels = []
for unit in dev_units:
	best_model = DegModel()
	best_model.load_state_dict(
		torch.load(DEGR_MODEL_DIR/'states'/perform_name/f'unit_{unit}'/ "best_model.pt")
	)
	degmodels.append(best_model) 
print(f'Number of component models: {len(degmodels)}')

FileNotFoundError: [Errno 2] No such file or directory: 'experiments/DS06/opcond_q0-1/estimation_thr1.0/gamma/states/SmHPC/unit_1/best_model.pt'

## Load test data

In [ ]:
hi_df = pd.read_csv(ESTIMATION_DIR/'data_test.csv')
test_units = hi_df['unit'].astype(int).unique().tolist()
test_units

[10, 11, 12, 13, 14, 15]

## Prepare data

In [ ]:
performs = {unit: hi_df[hi_df['unit']==unit][perform_name].values for unit in test_units} 
time = {unit: hi_df[hi_df['unit']==unit]['cycle'].values for unit in test_units}

test_data = {unit: torch.tensor(np.stack([t_data, s_data],axis=1),dtype=torch.float32)
             for t_data, s_data, unit in zip(time.values(), performs.values(), test_units)}

## Load PF-net

In [ ]:
ckpt = torch.load(performnet_dir/'checkpoint_best.pt', weights_only=False)
net = ParticleFilterMLP(state_dim=DegModel.state_dim(), hidden_dims=HIDDEN_DIMS,
                        activation=lambda : nn.LeakyReLU(LEAKY_SLOPE))
net.load_state_dict(ckpt['model_state'])

net.eval()   

ParticleFilterMLP(
  (net): Sequential(
    (0): Linear(in_features=2, out_features=256, bias=True)
    (1): LeakyReLU(negative_slope=0.05)
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): LeakyReLU(negative_slope=0.05)
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): LeakyReLU(negative_slope=0.05)
    (6): Linear(in_features=128, out_features=64, bias=True)
    (7): LeakyReLU(negative_slope=0.05)
    (8): Linear(in_features=64, out_features=32, bias=True)
    (9): LeakyReLU(negative_slope=0.05)
    (10): Linear(in_features=32, out_features=14, bias=True)
  )
)

## Run Particle Filter with (leraned net)

Save video

In [ ]:
t_grid = np.linspace(0.1, 100, 80)
s_grid = np.linspace(0.0, 1.0, 60)

for unit in test_units:
    # if unit != 10:
    #     continue
    t_data = test_data[unit][:, 0]
    s_data = test_data[unit][:, 1]
    t_data_np = t_data.cpu().numpy()
    s_data_np = s_data.cpu().numpy()

    pf = ParticleFilter(
        base_models=degmodels,
        net=net,  
        n_particles=N_PARTICLES,
    ).eval()

    frames = []
    for k in range(PREDICTION_START_IDX, len(t_data)):
        pf.step(
            t_obs=t_data[:k+1],
            s_obs=s_data[:k+1],
        )
        lower, mean, upper = pf.mixture.uncertainty_interval(s=torch.zeros(1), level=UNCERTAINTY_LEVEL)
        # --- render frame ---
        fig, ax = plt.subplots(figsize=(10, 8))
  
        create_pf_prediciton_frame(
            ax,
            pf,
            t_grid,
            s_grid,
            t_data_np,
            s_data_np,
            pred_interval=(lower.item(), mean.item(), upper.item()),
            conf_level=UNCERTAINTY_LEVEL,
            current_step = k,
            dist_plot_mean=True,
        )
  
        # create frame
        fig.canvas.draw()
        frame = np.asarray(fig.canvas.renderer.buffer_rgba())
        plt.close(fig)
        frames.append(frame)
  
    ## Save video
    video_path = pred_dir/f"{unit}-test.mp4"

    with imageio.get_writer(video_path, fps=8, macro_block_size=1) as writer:
        for frame in frames:
            writer.append_data(frame)

    print(f"🎬 Video saved to {video_path}")


🎬 Video saved to experiments/DS03/opcond_q0-1/estimation_thr0.1/gamma/net256x256x128x64x32_leaky0.05_npar2000_losswin5/SmHPC/pred_ulevel0.95/10-test.mp4
🎬 Video saved to experiments/DS03/opcond_q0-1/estimation_thr0.1/gamma/net256x256x128x64x32_leaky0.05_npar2000_losswin5/SmHPC/pred_ulevel0.95/11-test.mp4
🎬 Video saved to experiments/DS03/opcond_q0-1/estimation_thr0.1/gamma/net256x256x128x64x32_leaky0.05_npar2000_losswin5/SmHPC/pred_ulevel0.95/12-test.mp4
🎬 Video saved to experiments/DS03/opcond_q0-1/estimation_thr0.1/gamma/net256x256x128x64x32_leaky0.05_npar2000_losswin5/SmHPC/pred_ulevel0.95/13-test.mp4
🎬 Video saved to experiments/DS03/opcond_q0-1/estimation_thr0.1/gamma/net256x256x128x64x32_leaky0.05_npar2000_losswin5/SmHPC/pred_ulevel0.95/14-test.mp4
🎬 Video saved to experiments/DS03/opcond_q0-1/estimation_thr0.1/gamma/net256x256x128x64x32_leaky0.05_npar2000_losswin5/SmHPC/pred_ulevel0.95/15-test.mp4
